# Train VP-SDE Score Model with Clean Coordinate Channels

Separate score-based variant. Physical velocity channels are noised by the VP SDE; coordinate channels `(x, y)` are concatenated to the model input at every time and are never noised.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

## Clone repo and install dependencies

In [ ]:
import subprocess

REPO_URL = "https://github.com/SadreevAmir/DL_final_project.git"
REPO_DIR = "dl_final_project_score_training"

subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
!pip install -q -r requirements.txt

## Dataset source

In [ ]:
DATA_SOURCE = "https://disk.yandex.ru/d/rrjDGzzX5cfFnA"
assert DATA_SOURCE

## Configure score training

Defaults use normal full-pass epochs: `batches_per_epoch=0` and `val_batches=0`. Batch size and U-Net widths match the DDPM coordinate-channel baseline: batch `128`, diffusers.UNet2DModel channels `(96, 192, 384)`, 3 residual blocks per level, attention on lower resolutions, AdamW with weight decay `1e-3`, linear LR decay, and VP cosine SDE. Continuous VP-SDE `t in [0, 1]` is passed to diffusers as `t * 999`.

In [ ]:
from score_training import ScoreTrainConfig, prepare_score_dataset, train_score_model

config = ScoreTrainConfig(
    data_source=DATA_SOURCE,
    output_dir="runs_score",
    cache_dir="data/download_cache",
    stats_cache_path="data/download_cache/kolmogorov_velocity_256_to_64_train_stats.json",
    force_recompute_stats=False,
    dataset_tag="kolmogorov_velocity_256_to_64_coords",
    epochs=1024,
    batches_per_epoch=0,
    val_batches=0,
    batch_size=128,
    val_batch_size=128,
    num_workers=4,
    lr=2.0e-4,
    weight_decay=1.0e-3,
    sample_steps=256,
    sample_count=32,
    sample_every_epochs=1,
    display_samples_in_notebook=True,
    channels_per_level="96,192,384",
    num_res_blocks=3,
    attention_head_dim=32,
    time_embedding_scale=999.0,
    precision="bf16",
    compile_model=False,
    save_last_every_epochs=10,
    download_best_in_colab=False,
)

config

## Download and load dataset

This loads only physical data channels into RAM and normalizes them. Coordinate channels are created later on GPU and are not normalized/noised.

In [ ]:
dataset = prepare_score_dataset(config)
print("Train samples:", len(dataset.train))
print("Val samples:", len(dataset.val))
print("Stats cache:", dataset.stats.stats_cache_path)

## Start score training

In [ ]:
best_checkpoint = train_score_model(config, dataset=dataset)
print("Best checkpoint:", best_checkpoint)

## Inspect history and latest sample

In [ ]:
import json
from pathlib import Path
from IPython.display import display
from PIL import Image

run_dir = Path(best_checkpoint).parents[1]
history = json.loads((run_dir / "history.json").read_text())["history"]
print("Run dir:", run_dir)
print(json.dumps(history[-5:], indent=2))
latest = sorted((run_dir / "samples").glob("*.png"))[-1]
print(latest)
display(Image.open(latest))